# Data Preparation

This notebook documents all preprocessing steps applied to the two suicide rate datasets before analysis and modeling. Each transformation is justified by findings from the literature review.

**Datasets:**
- `suicide_rates_1990-2022.csv` — Granular (by country, year, sex, age group, generation)
- `age_std_suicide_rates_1990-2022.csv` — Age-standardized (by country, year, sex)

**Steps covered:**
1. Duplicate removal
2. Handling missing values (median imputation)
3. Filtering out unknown sex records
4. Data type casting (categorical, ordered categorical)
5. Log-transformation of InflationRate
6. Income-level stratification (World Bank classification)
7. Feature engineering — temporal lags (1yr, 3yr, 5yr)
8. Normalization of economic features for modeling
9. Export cleaned datasets

In [1]:
%pip install scikit-learn
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load both datasets
df_granular = pd.read_csv('suicide_rates_1990-2022.csv')
df_agestd = pd.read_csv('age_std_suicide_rates_1990-2022.csv')

print('Granular dataset shape:', df_granular.shape)
print('Age-standardized dataset shape:', df_agestd.shape)

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\SebastianStanik\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Granular dataset shape: (118560, 18)
Age-standardized dataset shape: (5928, 17)


---
## Step 1: Duplicate Removal

The granular dataset contains approximately 19% duplicate rows (22,584 rows). These duplicates would inflate suicide counts and bias statistical analysis. The age-standardized dataset has no duplicates.

In [2]:
# Check how many duplicates exist, then remove them
print('--- Granular Dataset ---')
print(f'Rows before: {len(df_granular):,}')
print(f'Duplicate rows: {df_granular.duplicated().sum():,}')

# drop_duplicates() removes exact duplicate rows, reset_index cleans up the index
df_granular = df_granular.drop_duplicates().reset_index(drop=True)

print(f'Rows after: {len(df_granular):,}')
print(f'Rows removed: {118560 - len(df_granular):,}')

print('\n--- Age-Standardized Dataset ---')
print(f'Duplicate rows: {df_agestd.duplicated().sum()} (no action needed)')

--- Granular Dataset ---
Rows before: 118,560
Duplicate rows: 22,584
Rows after: 95,976
Rows removed: 22,584

--- Age-Standardized Dataset ---
Duplicate rows: 0 (no action needed)


---
## Step 2: Filter Out Unknown Sex Records

Both datasets contain records where Sex = "Unknown". These rows lack demographic specificity needed for the regression analysis (Meda et al., 2022, found male populations are most sensitive to economic shocks). Removing them ensures clean male/female stratification.

In [3]:
# Check sex distribution before filtering
print('--- Granular Dataset: Sex distribution ---')
print(df_granular['Sex'].value_counts())

print('\n--- Age-Standardized Dataset: Sex distribution ---')
print(df_agestd['Sex'].value_counts())

# Remove rows where Sex is "Unknown" since we need clean male/female groups
df_granular = df_granular[df_granular['Sex'] != 'Unknown'].reset_index(drop=True)
df_agestd = df_agestd[df_agestd['Sex'] != 'Unknown'].reset_index(drop=True)

print(f'\nGranular after filtering: {len(df_granular):,} rows')
print(f'Age-standardized after filtering: {len(df_agestd):,} rows')

--- Granular Dataset: Sex distribution ---
Sex
Male       48912
Female     46077
Unknown      987
Name: count, dtype: int64

--- Age-Standardized Dataset: Sex distribution ---
Sex
Male       2916
Female     2916
Unknown      96
Name: count, dtype: int64

Granular after filtering: 94,989 rows
Age-standardized after filtering: 5,832 rows


---
## Step 3: Handle Missing Values

Missing values range from 0.02% to 12.2% across columns. Since the maximum is well below 50%, median imputation is viable. Median is chosen over mean because economic indicators like GDP and GNI are heavily right-skewed.

Imputation is performed **grouped by country** to preserve each nation's economic context rather than applying a global median.

In [4]:
# Define which columns need imputation in each dataset
econ_cols_granular = ['GDP', 'GDPPerCapita', 'GrossNationalIncome', 'GNIPerCapita',
                      'InflationRate', 'EmploymentPopulationRatio']
rate_cols_granular = ['SuicideCount', 'CauseSpecificDeathPercentage', 'DeathRatePer100K', 'Population']

econ_cols_agestd = ['GDP', 'GDPPerCapita', 'GNI', 'GNIPerCapita',
                    'InflationRate', 'EmploymentPopulationRatio']
rate_cols_agestd = ['CauseSpecificDeathPercentage', 'StdDeathRate', 'DeathRatePer100K', 'Population']

def show_missing(df, label):
    """Helper to display missing value counts and percentages."""
    miss = df.isnull().sum()
    miss_pct = (miss / len(df) * 100).round(2)
    table = pd.DataFrame({'Missing': miss, '%': miss_pct})
    table = table[table['Missing'] > 0]
    if len(table) == 0:
        print(f'{label}: No missing values')
    else:
        print(f'\n{label} — Missing values:')
        print(table)
    return table

print('=== BEFORE IMPUTATION ===')
show_missing(df_granular, 'Granular')
show_missing(df_agestd, 'Age-Standardized')

=== BEFORE IMPUTATION ===

Granular — Missing values:
                              Missing      %
SuicideCount                      292   0.31
CauseSpecificDeathPercentage     3670   3.86
DeathRatePer100K                 7256   7.64
Population                       4489   4.73
GDP                              5675   5.97
GDPPerCapita                     5675   5.97
GrossNationalIncome              7969   8.39
GNIPerCapita                     8610   9.06
InflationRate                   11503  12.11
EmploymentPopulationRatio        6879   7.24

Age-Standardized — Missing values:
                           Missing      %
StdDeathRate                   112   1.92
DeathRatePer100K               112   1.92
Population                     296   5.08
GDP                            362   6.21
GDPPerCapita                   362   6.21
GNI                            498   8.54
GNIPerCapita                   538   9.22
InflationRate                  700  12.00
EmploymentPopulationRatio      556   

,Missing,%
StdDeathRate,112,1.92
DeathRatePer100K,112,1.92
Population,296,5.08
GDP,362,6.21
GDPPerCapita,362,6.21
GNI,498,8.54
GNIPerCapita,538,9.22
InflationRate,700,12.00
EmploymentPopulationRatio,556,9.53


In [5]:
def median_impute_by_country(df, columns, country_col='CountryName'):
    """
    Fill missing values with the median for each country.
    If a country has no data at all for a column, fall back to the global median.
    We use median instead of mean because economic data tends to be skewed.
    """
    for col in columns:
        if df[col].isnull().sum() == 0:
            continue
        # First try: fill with that country's median
        country_medians = df.groupby(country_col)[col].transform('median')
        df[col] = df[col].fillna(country_medians)
        # Fallback: if a country had ALL missing values, use the global median
        global_median = df[col].median()
        df[col] = df[col].fillna(global_median)
    return df

# Run imputation on both datasets
all_impute_granular = econ_cols_granular + rate_cols_granular
all_impute_agestd = econ_cols_agestd + rate_cols_agestd

df_granular = median_impute_by_country(df_granular, all_impute_granular)
df_agestd = median_impute_by_country(df_agestd, all_impute_agestd)

print('=== AFTER IMPUTATION ===')
show_missing(df_granular, 'Granular')
show_missing(df_agestd, 'Age-Standardized')

=== AFTER IMPUTATION ===
Granular: No missing values
Age-Standardized: No missing values


,Missing,%


---
## Step 4: Cast Data Types

- **Sex** → unordered `Categorical`
- **AgeGroup** → ordered `Categorical` (ensures correct sorting: 0-14 < 15-24 < ... < 75+)
- **Generation** → ordered `Categorical` (chronological order)

This reduces memory usage and ensures correct ordering in visualizations and grouped analyses.

In [6]:
# Set up proper category ordering so age groups and generations sort correctly
age_order = ['0-14 years', '15-24 years', '25-34 years', '35-54 years',
             '55-74 years', '75+ years', 'Unknown']

gen_order = ['G.I. Generation', 'Silent Generation', 'Baby Boomers',
             'Generation X', 'Millennials', 'Generation Z', 'Generation Alpha']

# Cast to categorical types (ordered=True means they sort in the right order)
df_granular['Sex'] = pd.Categorical(df_granular['Sex'])
df_granular['AgeGroup'] = pd.Categorical(df_granular['AgeGroup'],
                                          categories=age_order, ordered=True)
df_granular['Generation'] = pd.Categorical(df_granular['Generation'],
                                            categories=gen_order, ordered=True)

df_agestd['Sex'] = pd.Categorical(df_agestd['Sex'])

print('Granular dtypes:')
print(df_granular[['Sex', 'AgeGroup', 'Generation']].dtypes)
print('\nAge-standardized dtypes:')
print(df_agestd[['Sex']].dtypes)

Granular dtypes:
Sex           category
AgeGroup      category
Generation    category
dtype: object

Age-standardized dtypes:
Sex    category
dtype: object


---
## Step 5: Log-Transform InflationRate

InflationRate has extreme values (up to 4,734.91%) reflecting real-world hyperinflation events. These extreme outliers would distort regression coefficients. A `log1p` transformation (log(1 + x)) compresses the scale while preserving the signal and handling the few negative values gracefully.

**Justification:** The lit review notes these values will be "kept to capture the 'cost of living' stressor but will be log-transformed in the regression model to prevent them from distorting the results."

In [7]:
# Log-transform inflation to compress extreme outliers like hyperinflation (up to 4734%)
# Using log1p (log of 1+x) which handles zero values gracefully
print('InflationRate — Before transformation:')
print(df_agestd['InflationRate'].describe().round(2))

for df_name, df in [('Granular', df_granular), ('Age-Standardized', df_agestd)]:
    min_val = df['InflationRate'].min()
    if min_val < 0:
        # Shift negative values up so everything is positive before taking log
        df['InflationRate_Log'] = np.log1p(df['InflationRate'] - min_val)
    else:
        df['InflationRate_Log'] = np.log1p(df['InflationRate'])

print('\nInflationRate_Log — After transformation (Age-Standardized):')
print(df_agestd['InflationRate_Log'].describe().round(4))

InflationRate — Before transformation:
count    5832.00
mean       18.73
std       154.18
min       -10.63
25%         1.63
50%         3.10
75%         6.35
max      4734.91
Name: InflationRate, dtype: float64

InflationRate_Log — After transformation (Age-Standardized):
count    5832.0000
mean        2.8329
std         0.5689
min         0.0000
25%         2.5848
50%         2.6897
75%         2.8894
max         8.4652
Name: InflationRate_Log, dtype: float64


---
## Step 6: Income-Level Stratification

Countries are classified into **High-Income** and **Low/Middle-Income** groups based on their median GNI per capita across all years. This uses the World Bank threshold of approximately $13,845 (2022 classification).

**Justification:** Lyu et al. (2025) demonstrated that economic growth is protective in low-income nations but associated with *higher* suicide rates in high-income nations (the "Protective Paradox"). Er et al. (2023) found economic uncertainty significantly increases suicide risk only in high-income countries. This stratification directly addresses **Research Question 3**.

In [8]:
# Split countries into High-Income vs Low/Middle-Income using World Bank threshold
HIGH_INCOME_THRESHOLD = 13845  # 2022 World Bank GNI per capita cutoff

def classify_income(df, gni_col='GNIPerCapita'):
    """Classify each country based on its median GNI per capita across all years."""
    country_median_gni = df.groupby('CountryName')[gni_col].median()
    high_income_countries = country_median_gni[country_median_gni >= HIGH_INCOME_THRESHOLD].index
    df['IncomeGroup'] = df['CountryName'].apply(
        lambda x: 'High-Income' if x in high_income_countries else 'Low/Middle-Income'
    )
    return df

df_granular = classify_income(df_granular)
df_agestd = classify_income(df_agestd)

print('--- Granular: Income Group distribution ---')
print(df_granular['IncomeGroup'].value_counts())
print(f'\nHigh-Income countries: {df_granular[df_granular["IncomeGroup"]=="High-Income"]["CountryName"].nunique()}')
print(f'Low/Middle-Income countries: {df_granular[df_granular["IncomeGroup"]=="Low/Middle-Income"]["CountryName"].nunique()}')

print('\n--- Age-Standardized: Income Group distribution ---')
print(df_agestd['IncomeGroup'].value_counts())

--- Granular: Income Group distribution ---
IncomeGroup
High-Income          57176
Low/Middle-Income    37813
Name: count, dtype: int64

High-Income countries: 66
Low/Middle-Income countries: 51

--- Age-Standardized: Income Group distribution ---
IncomeGroup
High-Income          3462
Low/Middle-Income    2370
Name: count, dtype: int64


---
## Step 7: Feature Engineering — Temporal Lags

Economic shocks do not instantaneously result in public health crises. Lagged features capture the delayed impact of economic changes on suicide rates.

**Justification:** Er et al. (2023) demonstrated that the psychological impact of economic uncertainty operates on a delay. Reeves et al. (2014) showed that unemployment spikes are followed by a *delayed, prolonged* rise in suicide rates.

We create **1-year, 3-year, and 5-year lagged** versions of:
- GDP per capita year-over-year growth rate
- InflationRate (log-transformed)
- EmploymentPopulationRatio

In [9]:
def add_temporal_lags(df, lag_cols, lags=[1, 3, 5], group_cols=['CountryName', 'Sex']):
    """
    Create lagged versions of features (1yr, 3yr, 5yr behind).
    Economic shocks don't affect suicide rates instantly - there's a delay.
    """
    df = df.sort_values(group_cols + ['Year']).reset_index(drop=True)
    
    # Year-over-year GDP growth rate (how much did GDP change from last year?)
    df['GDPPerCapita_Growth'] = df.groupby(group_cols)['GDPPerCapita'].pct_change() * 100
    
    # Create lagged versions: shift(1) = last year, shift(3) = 3 years ago, etc.
    for col in lag_cols:
        for lag in lags:
            lag_col_name = f'{col}_Lag{lag}'
            df[lag_col_name] = df.groupby(group_cols)[col].shift(lag)
    
    return df

# Features to create lags for
lag_features = ['GDPPerCapita_Growth', 'InflationRate_Log', 'EmploymentPopulationRatio']

# Apply to both datasets
df_agestd = add_temporal_lags(df_agestd, lag_features,
                              group_cols=['CountryName', 'Sex'])
df_granular = add_temporal_lags(df_granular, lag_features,
                                group_cols=['CountryName', 'Sex', 'AgeGroup'])

# Show the new columns and how many non-null values each has
new_cols = [c for c in df_agestd.columns if 'Lag' in c or c == 'GDPPerCapita_Growth']
print('New engineered features:')
for col in new_cols:
    non_null = df_agestd[col].notna().sum()
    print(f'  {col}: {non_null:,} non-null values')

New engineered features:
  GDPPerCapita_Growth: 5,598 non-null values
  GDPPerCapita_Growth_Lag1: 5,366 non-null values
  GDPPerCapita_Growth_Lag3: 4,910 non-null values
  GDPPerCapita_Growth_Lag5: 4,466 non-null values
  InflationRate_Log_Lag1: 5,598 non-null values
  InflationRate_Log_Lag3: 5,134 non-null values
  InflationRate_Log_Lag5: 4,686 non-null values
  EmploymentPopulationRatio_Lag1: 5,598 non-null values
  EmploymentPopulationRatio_Lag3: 5,134 non-null values
  EmploymentPopulationRatio_Lag5: 4,686 non-null values


---
## Step 8: Normalize Economic Features for Modeling

GDP, GNI, and per-capita values span vastly different scales (e.g., GDP in billions vs. InflationRate in single digits). StandardScaler (z-score normalization) is applied to ensure all features contribute equally to the regression model.

Normalized columns are stored separately with a `_Scaled` suffix to preserve the original values for interpretability.

In [10]:
# Normalize economic features so they're all on the same scale (mean=0, std=1)
# Important because GDP is in billions while inflation is in single digits
features_to_scale = ['GDPPerCapita', 'GNIPerCapita', 'InflationRate_Log',
                     'EmploymentPopulationRatio']

scaler = StandardScaler()

scaled_values = scaler.fit_transform(df_agestd[features_to_scale])
for i, col in enumerate(features_to_scale):
    df_agestd[f'{col}_Scaled'] = scaled_values[:, i]

# Verify: scaled features should have mean ~0 and std ~1
print('Scaled feature statistics (should be ~mean=0, std=1):')
scaled_cols = [f'{c}_Scaled' for c in features_to_scale]
print(df_agestd[scaled_cols].describe().round(4).loc[['mean', 'std']])

Scaled feature statistics (should be ~mean=0, std=1):
      GDPPerCapita_Scaled  GNIPerCapita_Scaled  InflationRate_Log_Scaled  \
mean               0.0000              -0.0000                    0.0000   
std                1.0001               1.0001                    1.0001   

      EmploymentPopulationRatio_Scaled  
mean                            0.0000  
std                             1.0001  


---
## Step 9: Final Dataset Summary & Export

Export the cleaned and engineered datasets for use in the analysis and modeling notebooks.

In [11]:
# Final summary of what we have after all the cleaning and engineering
print('=== FINAL DATASET SUMMARY ===')
print(f'\nGranular dataset: {df_granular.shape[0]:,} rows × {df_granular.shape[1]} columns')
print(f'Age-standardized dataset: {df_agestd.shape[0]:,} rows × {df_agestd.shape[1]} columns')

print('\n--- Granular columns ---')
print(list(df_granular.columns))

print('\n--- Age-standardized columns ---')
print(list(df_agestd.columns))

=== FINAL DATASET SUMMARY ===

Granular dataset: 94,989 rows × 30 columns
Age-standardized dataset: 5,832 rows × 33 columns

--- Granular columns ---
['RegionCode', 'RegionName', 'CountryCode', 'CountryName', 'Year', 'Sex', 'AgeGroup', 'Generation', 'SuicideCount', 'CauseSpecificDeathPercentage', 'DeathRatePer100K', 'Population', 'GDP', 'GDPPerCapita', 'GrossNationalIncome', 'GNIPerCapita', 'InflationRate', 'EmploymentPopulationRatio', 'InflationRate_Log', 'IncomeGroup', 'GDPPerCapita_Growth', 'GDPPerCapita_Growth_Lag1', 'GDPPerCapita_Growth_Lag3', 'GDPPerCapita_Growth_Lag5', 'InflationRate_Log_Lag1', 'InflationRate_Log_Lag3', 'InflationRate_Log_Lag5', 'EmploymentPopulationRatio_Lag1', 'EmploymentPopulationRatio_Lag3', 'EmploymentPopulationRatio_Lag5']

--- Age-standardized columns ---
['RegionCode', 'RegionName', 'CountryCode', 'CountryName', 'Year', 'Sex', 'SuicideCount', 'CauseSpecificDeathPercentage', 'StdDeathRate', 'DeathRatePer100K', 'Population', 'GDP', 'GDPPerCapita', 'GNI', '

In [12]:
# Save cleaned datasets to CSV for the analysis and modeling notebooks
df_granular.to_csv('suicide_rates_cleaned.csv', index=False)
df_agestd.to_csv('age_std_suicide_rates_cleaned.csv', index=False)

print('Cleaned datasets exported:')
print('  -> suicide_rates_cleaned.csv')
print('  -> age_std_suicide_rates_cleaned.csv')

Cleaned datasets exported:
  -> suicide_rates_cleaned.csv
  -> age_std_suicide_rates_cleaned.csv


---
## Summary of Transformations Applied

| Step | Transformation | Justification |
|------|---------------|---------------|
| 1 | Removed 22,584 duplicate rows (granular) | Prevent inflated counts and biased statistics |
| 2 | Filtered out Sex = "Unknown" | Clean male/female stratification for demographic analysis |
| 3 | Median imputation (by country) | Handles right-skewed economic data without mean distortion |
| 4 | Categorical type casting | Correct ordering for AgeGroup/Generation; memory optimization |
| 5 | Log-transform InflationRate | Compress hyperinflation extremes for regression stability |
| 6 | Income-level stratification | Test the Protective Paradox (Lyu et al., 2025; Er et al., 2023) |
| 7 | 1/3/5-year temporal lags | Capture delayed economic impact on suicide (Reeves et al., 2014) |
| 8 | StandardScaler normalization | Equalize feature scales for regression modeling |